# 🔎 Quill Agent — Draft

Quill is the critique agent in the Seesaw pipeline. It takes an **ExperimentBundle** from Lens
and produces a structured **CritiqueReport** — asking whether the experiments were well-designed,
whether the conclusions hold, and what's missing.

### Pipeline position
```
Scout → [Research Plan] → Lens → [ExperimentBundle] → Quill → [CritiqueReport]
                                        ▲                              │
                                        └── follow-up specs (optional)─┘
```

### What Quill checks
1. **Validity** — is each experiment's methodology sound?
2. **Support** — do the conclusions follow from the data?
3. **Confounds** — what alternative explanations weren't ruled out?
4. **Gaps** — which experiments are conspicuously absent given the research question?
5. **Follow-ups** — concrete experiment specs Lens could run next

### Architecture
Three focused LangGraph nodes:
1. **`critique_experiments`** — per-experiment assessment (validity, confounds, alternative explanations)
2. **`identify_gaps`** — what the research question demands that wasn't tested
3. **`generate_followups`** — structured `FollowUpSpec` objects Lens can execute directly
4. **`save_critique`** — writes `critique.json` + `critique.md`

## 1. Setup

In [ ]:
%pip install -q \
    langchain-anthropic \
    langgraph \
    pydantic

In [ ]:
import json
import os
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Literal, TypedDict

from IPython.display import Markdown, display
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel

print("✅ Imports OK")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

# One model, analytical temperature — Quill is a critic, not a creative writer
model = ChatAnthropic(
    model="claude-opus-4-5",
    temperature=0.2,
    api_key=ANTHROPIC_API_KEY,
)

print("✅ Model ready")

## 2. Schemas

In [ ]:
# ── Lens output schemas (mirrors lens_draft.ipynb) ────────────────────────────
@dataclass
class ExperimentResult:
    name: str
    tool: str
    model_name: str
    prompts: list[str]
    summary: str = ""
    plot_paths: list[str] = field(default_factory=list)
    data: dict = field(default_factory=dict)
    status: Literal["success", "failed", "timeout"] = "success"
    error: str | None = None


@dataclass
class ExperimentBundle:
    research_question: str
    model_name: str
    results: list[ExperimentResult]

    @property
    def successful(self) -> list[ExperimentResult]:
        return [r for r in self.results if r.status == "success"]

    @property
    def failed(self) -> list[ExperimentResult]:
        return [r for r in self.results if r.status != "success"]

    @classmethod
    def from_json(cls, path: str | Path) -> "ExperimentBundle":
        with open(path) as f:
            raw = json.load(f)
        results = [ExperimentResult(**r) for r in raw["results"]]
        return cls(
            research_question=raw["research_question"],
            model_name=raw["model_name"],
            results=results,
        )


# ── Quill output schemas ───────────────────────────────────────────────────────
class ExperimentCritique(BaseModel):
    """Per-experiment assessment."""
    experiment_name: str
    validity: Literal["strong", "moderate", "weak"]
    conclusions_supported: bool
    issues: list[str]                  # confounds, methodology problems
    alternative_explanations: list[str]  # other ways to read the results
    positive_aspects: list[str]        # what the experiment does well


class ResearchGap(BaseModel):
    """An experiment that should have been run but wasn't."""
    description: str
    severity: Literal["critical", "important", "minor"]
    why_needed: str     # what question it would answer
    suggested_tool: str  # which Lens tool would address it


class FollowUpSpec(BaseModel):
    """Concrete experiment spec Lens can execute directly."""
    name: str
    tool: str           # logit_lens | attention_pattern | ablation | activation_patching | direct_logit_attribution
    rationale: str      # why this follow-up is needed
    what_to_measure: str
    hypothesis: str
    priority: Literal["high", "medium", "low"]


# ── Intermediate structured outputs for each node ─────────────────────────────
class ExperimentCritiquesOutput(BaseModel):
    critiques: list[ExperimentCritique]
    overall_assessment: Literal["strong", "moderate", "weak"]
    overall_summary: str


class GapsOutput(BaseModel):
    gaps: list[ResearchGap]
    coverage_verdict: str  # 1-2 sentences on how well the experiments cover the question


class FollowUpsOutput(BaseModel):
    followups: list[FollowUpSpec]


# ── Final report ───────────────────────────────────────────────────────────────
@dataclass
class CritiqueReport:
    research_question: str
    model_name: str
    overall_assessment: Literal["strong", "moderate", "weak"]
    overall_summary: str
    coverage_verdict: str
    critiques: list[ExperimentCritique]
    gaps: list[ResearchGap]
    followups: list[FollowUpSpec]
    generated_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def to_dict(self) -> dict:
        return {
            "research_question": self.research_question,
            "model_name": self.model_name,
            "overall_assessment": self.overall_assessment,
            "overall_summary": self.overall_summary,
            "coverage_verdict": self.coverage_verdict,
            "generated_at": self.generated_at,
            "critiques": [c.model_dump() for c in self.critiques],
            "gaps": [g.model_dump() for g in self.gaps],
            "followups": [f.model_dump() for f in self.followups],
        }


print("✅ Schemas defined")

## 3. Context Helpers

In [ ]:
def format_result_for_context(result: ExperimentResult) -> str:
    lines = [
        f"### Experiment: {result.name}",
        f"- **Tool**: `{result.tool}`",
        f"- **Model**: {result.model_name}",
        f"- **Status**: {result.status}",
        f"- **Prompts used**: {len(result.prompts)} (e.g. `{result.prompts[0][:80]}`)",
        "",
        f"**Lens summary**: {result.summary or '(no summary)'}",
    ]
    if result.data:
        lines += ["", "**Key measurements**:"]
        for k, v in list(result.data.items())[:8]:
            lines.append(f"  - `{k}`: {str(v)[:200]}")
    if result.error:
        lines.append(f"**Error**: {result.error}")
    return "\n".join(lines)


def bundle_to_context(bundle: ExperimentBundle) -> str:
    parts = [f"Research question: {bundle.research_question}",
             f"Model: {bundle.model_name}",
             f"Experiments: {len(bundle.successful)} successful, {len(bundle.failed)} failed",
             ""]
    parts += [format_result_for_context(r) for r in bundle.successful]
    if bundle.failed:
        parts.append("\n### Failed experiments")
        for r in bundle.failed:
            parts.append(f"- `{r.name}` ({r.tool}): {r.status} — {r.error or 'no error message'}")
    return "\n\n".join(parts)


print("✅ Context helpers ready")

## 4. Prompts

In [ ]:
CRITIQUE_EXPERIMENTS_SYSTEM = """
You are a senior mechanistic interpretability researcher reviewing a set of experiments.

Your job is to critically evaluate each experiment on:

1. **Validity** — Is the methodology sound? Does the experiment actually measure what it claims to measure?
   - strong: clean setup, appropriate controls, unambiguous signal
   - moderate: mostly sound but with minor issues
   - weak: significant methodological problems

2. **Support** — Do the stated conclusions follow logically from the data? Be conservative.
   A high attention weight is not by itself causal evidence. A logit lens signal is not an ablation.

3. **Confounds** — What else could explain the result?
   Common confounds in mech interp:
   - Attention patterns ≠ information movement (value vectors matter too)
   - Small prompt sets may not generalise
   - Correlation between token positions confounds patching
   - Mean ablation may destroy more than just the target feature

4. **Alternative explanations** — How else might these results be interpreted?

5. **Positive aspects** — What does the experiment do well?

Be precise and concrete. Cite specific numbers from the data when making a claim.
Do not pad. If an experiment is genuinely strong, say so.
""".strip()


IDENTIFY_GAPS_SYSTEM = """
You are a senior mechanistic interpretability researcher.

Given a research question and a set of experiments that were run, identify what is MISSING.

Think about:
- What does the research question require to be answered definitively?
- Which claims in the experiment summaries are unsupported by the experiments run?
- What is the standard toolkit for this type of question? (e.g., for circuit identification:
  logit lens, activation patching, ablation, direct logit attribution, attention patterns)
- Are there experiments that would distinguish between competing explanations?
- Is the prompt set large/diverse enough to support generalisation?
- Were controls run? (e.g., random ablations, symmetric prompts, scrambled inputs)

Classify each gap by severity:
- critical: without this, the central claim cannot be made
- important: significantly strengthens or weakens the conclusion
- minor: useful but not essential

Be specific. Don't suggest vague follow-ups — name the exact experiment type and why it's needed.
""".strip()


GENERATE_FOLLOWUPS_SYSTEM = """
You are designing follow-up experiments for a mechanistic interpretability study.

Based on the identified gaps and experiment critiques, generate concrete follow-up experiment specs
that an automated agent (Lens) can execute.

Available tools:
- `logit_lens` — residual stream projections per layer/position
- `attention_pattern` — attention weights per head per layer
- `ablation` — zero/mean ablate a component and measure logit diff
- `activation_patching` — patch activations from a clean→corrupted run to localise causally
- `direct_logit_attribution` — decompose final logit into per-head contributions

For each follow-up:
- Choose the tool that most directly tests the gap or weakness
- Write a precise hypothesis (what result would confirm/disconfirm the claim?)
- Describe exactly what to measure
- Assign priority: high (critical gap), medium (important gap), low (minor gap)

Keep follow-ups actionable. Omit any that require data or infrastructure not available from
the existing ExperimentBundle and TransformerLens.
""".strip()


print("✅ Prompts ready")

## 5. LangGraph Workflow

```
critique_experiments
        │
        ▼
  identify_gaps
        │
        ▼
 generate_followups
        │
        ▼
   save_critique ──► END
```

Three independent analytical passes, each with its own focused prompt and structured output.
No loops — critique is deterministic given the bundle.

In [ ]:
# ── State ─────────────────────────────────────────────────────────────────────
class QuillState(TypedDict):
    # Input
    bundle: ExperimentBundle
    output_dir: Path

    # Node outputs
    critiques_output: ExperimentCritiquesOutput | None
    gaps_output: GapsOutput | None
    followups_output: FollowUpsOutput | None

    # Final
    critique_report: CritiqueReport | None
    critique_path: Path | None


print("✅ State defined")

In [ ]:
# ── Node 1: critique_experiments ──────────────────────────────────────────────
def critique_experiments(state: QuillState) -> dict:
    bundle = state["bundle"]
    print("\n🔬 [critique_experiments] Assessing each experiment...")

    ctx = bundle_to_context(bundle)
    prompt = f"""Please critique each of the following experiments.

{ctx}

Assess every successful experiment individually.
Then give an overall assessment of the experiment set as a whole."""

    structured = model.with_structured_output(ExperimentCritiquesOutput)
    output: ExperimentCritiquesOutput = structured.invoke([
        SystemMessage(content=CRITIQUE_EXPERIMENTS_SYSTEM),
        HumanMessage(content=prompt),
    ])

    print(f"  → Overall: {output.overall_assessment}")
    for c in output.critiques:
        supported = "✓" if c.conclusions_supported else "✗"
        print(f"  → {c.experiment_name}: {c.validity} | conclusions_supported={supported} | issues={len(c.issues)}")

    return {"critiques_output": output}


print("✅ critique_experiments defined")

In [ ]:
# ── Node 2: identify_gaps ─────────────────────────────────────────────────────
def identify_gaps(state: QuillState) -> dict:
    bundle = state["bundle"]
    critiques_output = state["critiques_output"]
    print("\n🕳️  [identify_gaps] Identifying missing experiments...")

    ctx = bundle_to_context(bundle)
    tools_used = list({r.tool for r in bundle.successful})
    critique_summary = "\n".join(
        f"- {c.experiment_name}: {c.validity}, issues: {'; '.join(c.issues[:2])}"
        for c in critiques_output.critiques
    )

    prompt = f"""Research question: {bundle.research_question}

Tools used so far: {tools_used}

Experiment critiques (summary):
{critique_summary}

Full experiment data:
{ctx}

What experiments are missing? What would a thorough investigation of this research question require
that wasn't done?"""

    structured = model.with_structured_output(GapsOutput)
    output: GapsOutput = structured.invoke([
        SystemMessage(content=IDENTIFY_GAPS_SYSTEM),
        HumanMessage(content=prompt),
    ])

    critical = [g for g in output.gaps if g.severity == "critical"]
    important = [g for g in output.gaps if g.severity == "important"]
    minor = [g for g in output.gaps if g.severity == "minor"]
    print(f"  → {len(output.gaps)} gaps: {len(critical)} critical, {len(important)} important, {len(minor)} minor")
    for g in output.gaps:
        print(f"     [{g.severity}] {g.description[:80]}")

    return {"gaps_output": output}


print("✅ identify_gaps defined")

In [ ]:
# ── Node 3: generate_followups ────────────────────────────────────────────────
def generate_followups(state: QuillState) -> dict:
    bundle = state["bundle"]
    critiques_output = state["critiques_output"]
    gaps_output = state["gaps_output"]
    print("\n🧪 [generate_followups] Designing follow-up experiments...")

    gaps_text = "\n".join(
        f"- [{g.severity}] {g.description} → needs: {g.suggested_tool} ({g.why_needed})"
        for g in gaps_output.gaps
    )
    weak_experiments = [
        c for c in critiques_output.critiques
        if c.validity in ("weak", "moderate") or not c.conclusions_supported
    ]
    weak_text = "\n".join(
        f"- {c.experiment_name}: {'; '.join(c.issues[:2])}"
        for c in weak_experiments
    ) or "(none — all experiments are sound)"

    prompt = f"""Research question: {bundle.research_question}
Model: {bundle.model_name}

Identified gaps:
{gaps_text}

Experiments needing follow-up (weak or unsupported conclusions):
{weak_text}

Design concrete follow-up experiments that Lens can run to address these gaps and weaknesses.
Focus on high-priority items first. Keep the total to 3-5 follow-ups."""

    structured = model.with_structured_output(FollowUpsOutput)
    output: FollowUpsOutput = structured.invoke([
        SystemMessage(content=GENERATE_FOLLOWUPS_SYSTEM),
        HumanMessage(content=prompt),
    ])

    by_priority = {"high": [], "medium": [], "low": []}
    for f in output.followups:
        by_priority[f.priority].append(f.name)
    print(f"  → {len(output.followups)} follow-ups:")
    for p, names in by_priority.items():
        if names:
            print(f"     {p}: {', '.join(names)}")

    return {"followups_output": output}


print("✅ generate_followups defined")

In [ ]:
# ── Node 4: save_critique ─────────────────────────────────────────────────────
def save_critique(state: QuillState) -> dict:
    bundle = state["bundle"]
    critiques_output = state["critiques_output"]
    gaps_output = state["gaps_output"]
    followups_output = state["followups_output"]
    output_dir = state["output_dir"]
    print("\n💾 [save_critique] Assembling and saving critique report...")

    report = CritiqueReport(
        research_question=bundle.research_question,
        model_name=bundle.model_name,
        overall_assessment=critiques_output.overall_assessment,
        overall_summary=critiques_output.overall_summary,
        coverage_verdict=gaps_output.coverage_verdict,
        critiques=critiques_output.critiques,
        gaps=gaps_output.gaps,
        followups=followups_output.followups,
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # ── Save JSON ──────────────────────────────────────────────────────────────
    json_path = output_dir / f"critique_{timestamp}.json"
    with open(json_path, "w") as f:
        json.dump(report.to_dict(), f, indent=2)

    # ── Save Markdown ──────────────────────────────────────────────────────────
    md_path = output_dir / f"critique_{timestamp}.md"
    with open(md_path, "w") as f:
        f.write(render_critique_markdown(report))

    print(f"  ✅ JSON  → {json_path}")
    print(f"  ✅ MD    → {md_path}")

    return {
        "critique_report": report,
        "critique_path": md_path,
    }


def render_critique_markdown(report: CritiqueReport) -> str:
    """Render a CritiqueReport to human-readable markdown."""
    SEVERITY_EMOJI = {"critical": "🔴", "important": "🟡", "minor": "🟢"}
    VALIDITY_EMOJI = {"strong": "✅", "moderate": "⚠️", "weak": "❌"}
    PRIORITY_EMOJI = {"high": "🔴", "medium": "🟡", "low": "🟢"}

    lines = [
        f"# Quill Critique Report",
        f"",
        f"> **Research question**: {report.research_question}",
        f"> **Model**: `{report.model_name}`",
        f"> **Generated**: {report.generated_at[:19]}",
        f"",
        f"---",
        f"",
        f"## Overall Assessment: {report.overall_assessment.upper()}",
        f"",
        report.overall_summary,
        f"",
        f"**Coverage**: {report.coverage_verdict}",
        f"",
        f"---",
        f"",
        f"## Experiment Critiques",
    ]

    for c in report.critiques:
        supported = "conclusions supported" if c.conclusions_supported else "**conclusions NOT supported**"
        lines += [
            f"",
            f"### {VALIDITY_EMOJI[c.validity]} {c.experiment_name}",
            f"- **Validity**: {c.validity} | {supported}",
        ]
        if c.positive_aspects:
            lines.append("- **Strengths**: " + "; ".join(c.positive_aspects))
        if c.issues:
            lines.append("- **Issues**:")
            for issue in c.issues:
                lines.append(f"  - {issue}")
        if c.alternative_explanations:
            lines.append("- **Alternative explanations**:")
            for alt in c.alternative_explanations:
                lines.append(f"  - {alt}")

    lines += [
        f"",
        f"---",
        f"",
        f"## Research Gaps",
    ]

    if not report.gaps:
        lines.append("\nNo significant gaps identified.")
    else:
        for g in sorted(report.gaps, key=lambda x: ["critical", "important", "minor"].index(x.severity)):
            lines += [
                f"",
                f"### {SEVERITY_EMOJI[g.severity]} {g.description}",
                f"- **Severity**: {g.severity}",
                f"- **Why needed**: {g.why_needed}",
                f"- **Suggested tool**: `{g.suggested_tool}`",
            ]

    lines += [
        f"",
        f"---",
        f"",
        f"## Suggested Follow-Up Experiments",
        f"",
        f"*These specs can be passed directly to Lens for execution.*",
    ]

    if not report.followups:
        lines.append("\nNo follow-up experiments suggested.")
    else:
        for fu in sorted(report.followups, key=lambda x: ["high", "medium", "low"].index(x.priority)):
            lines += [
                f"",
                f"### {PRIORITY_EMOJI[fu.priority]} {fu.name}",
                f"- **Tool**: `{fu.tool}`",
                f"- **Priority**: {fu.priority}",
                f"- **Rationale**: {fu.rationale}",
                f"- **What to measure**: {fu.what_to_measure}",
                f"- **Hypothesis**: {fu.hypothesis}",
            ]

    return "\n".join(lines)


print("✅ save_critique + render_critique_markdown defined")

In [ ]:
# ── Build the graph ───────────────────────────────────────────────────────────
memory = MemorySaver()
builder = StateGraph(QuillState)

builder.add_node("critique_experiments", critique_experiments)
builder.add_node("identify_gaps",        identify_gaps)
builder.add_node("generate_followups",   generate_followups)
builder.add_node("save_critique",        save_critique)

builder.set_entry_point("critique_experiments")
builder.add_edge("critique_experiments", "identify_gaps")
builder.add_edge("identify_gaps",        "generate_followups")
builder.add_edge("generate_followups",   "save_critique")
builder.add_edge("save_critique",        END)

quill_graph = builder.compile(checkpointer=memory)

print("✅ Quill graph compiled")
print("   Nodes:", list(quill_graph.get_graph().nodes.keys()))

## 6. Sample ExperimentBundle (Mock Lens Output)

In production, Quill reads `lens/outputs/<run>/results_bundle.json`.
This mock bundle is intentionally designed with some weaknesses so the critique has something to find.

In [ ]:
MOCK_BUNDLE = ExperimentBundle(
    research_question="How does GPT-2 Small implement the Indirect Object Identification (IOI) task?",
    model_name="gpt2",
    results=[
        ExperimentResult(
            name="logit_lens_ioi",
            tool="logit_lens",
            model_name="gpt2",
            prompts=[
                "When Mary and John went to the store, John gave a drink to",
            ],  # only 1 prompt — weak generalisation
            summary=(
                "The logit lens reveals that the correct IO (Mary) emerges from layer 7, "
                "peaking at layer 9 with 72% probability. Earlier layers show near-uniform "
                "distributions."
            ),
            data={
                "peak_layer": 9,
                "top_token_layer_9": "Mary",
                "top_token_prob_layer_9": 0.72,
                "baseline_prob_layer_0": 0.03,
            },
            status="success",
        ),
        ExperimentResult(
            name="attention_pattern_layer9",
            tool="attention_pattern",
            model_name="gpt2",
            prompts=[
                "When Mary and John went to the store, John gave a drink to",
            ],
            summary=(
                "Heads 9.6 and 9.9 attend strongly to Mary at the final token position. "
                "These are the name-mover heads identified by Wang et al. (2022). "
                "This confirms they are causally responsible for the IOI behaviour."
                # Note: attention pattern ≠ causal — this conclusion is unsupported without ablation
            ),
            data={
                "head_9_6_io_attention": 0.68,
                "head_9_9_io_attention": 0.81,
                "head_9_6_s_attention":  0.04,
                "head_9_9_s_attention":  0.06,
            },
            status="success",
        ),
        ExperimentResult(
            name="attention_pattern_layer3",
            tool="attention_pattern",
            model_name="gpt2",
            prompts=[
                "When Mary and John went to the store, John gave a drink to",
            ],
            summary=(
                "Head 3.0 attends from the second 'John' to the first 'John', "
                "consistent with duplicate token detection behaviour."
            ),
            data={
                "head_3_0_duplicate_token_attention": 0.55,
            },
            status="success",
        ),
        ExperimentResult(
            name="ablation_attempt",
            tool="ablation",
            model_name="gpt2",
            prompts=[],
            summary="",
            status="failed",
            error="Tool not yet implemented",
        ),
    ],
)

print(f"✅ Mock bundle — {len(MOCK_BUNDLE.successful)} successful, {len(MOCK_BUNDLE.failed)} failed")
print("   (Intentionally weak: 1 prompt, no ablation, attention pattern mistaken for causation)")

## 7. Run Quill

In [ ]:
OUTPUT_DIR = Path("../quill/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
config = {"configurable": {"thread_id": f"quill_run_{RUN_ID}"}}

initial_state: QuillState = {
    "bundle":            MOCK_BUNDLE,
    "output_dir":        OUTPUT_DIR,
    "critiques_output":  None,
    "gaps_output":       None,
    "followups_output":  None,
    "critique_report":   None,
    "critique_path":     None,
}

print(f"🚀 Starting Quill critique run: {RUN_ID}")
print(f"   Question: {MOCK_BUNDLE.research_question}")
print(f"   Experiments: {len(MOCK_BUNDLE.successful)} successful, {len(MOCK_BUNDLE.failed)} failed")

In [ ]:
for step in quill_graph.stream(initial_state, config=config, stream_mode="updates"):
    node_name = list(step.keys())[0]
    update    = list(step.values())[0]

    if node_name == "critique_experiments":
        out = update.get("critiques_output")
        if out:
            print(f"\n{'='*60}")
            print(f"🔬 CRITIQUE  |  overall={out.overall_assessment}  |  {len(out.critiques)} experiments assessed")

    elif node_name == "identify_gaps":
        out = update.get("gaps_output")
        if out:
            critical = sum(1 for g in out.gaps if g.severity == "critical")
            print(f"🕳️  GAPS     |  {len(out.gaps)} total  |  {critical} critical")

    elif node_name == "generate_followups":
        out = update.get("followups_output")
        if out:
            high = sum(1 for f in out.followups if f.priority == "high")
            print(f"🧪 FOLLOWUPS |  {len(out.followups)} total  |  {high} high priority")

    elif node_name == "save_critique":
        path = update.get("critique_path")
        print(f"\n{'='*60}")
        print(f"✅ DONE  |  {path}")

print("\n🏁 Quill complete")

## 8. Display Critique

In [ ]:
full_state = quill_graph.get_state(config).values
critique_path = full_state.get("critique_path")

if critique_path and Path(str(critique_path)).exists():
    with open(critique_path) as f:
        md = f.read()
    display(Markdown(md))
else:
    print("⚠️  Critique not found — check if Quill ran successfully")

In [ ]:
# ── Inspect follow-up specs (for feeding back into Lens) ─────────────────────
report = full_state.get("critique_report")

if report and report.followups:
    print(f"📋 {len(report.followups)} follow-up specs ready for Lens:\n")
    for fu in sorted(report.followups, key=lambda x: ["high", "medium", "low"].index(x.priority)):
        print(f"[{fu.priority.upper()}] {fu.name}")
        print(f"  tool:      {fu.tool}")
        print(f"  measure:   {fu.what_to_measure}")
        print(f"  hypothesis:{fu.hypothesis}")
        print()

## 9. Load from Real Lens Output (Optional)

In [ ]:
# LENS_OUTPUT = Path("../lens/outputs/results_bundle.json")
# if LENS_OUTPUT.exists():
#     real_bundle = ExperimentBundle.from_json(LENS_OUTPUT)
#     print(f"✅ {len(real_bundle.successful)} experiments loaded")
# To use: replace MOCK_BUNDLE with real_bundle in section 7
print("(Uncomment to load a real Lens bundle)")

## 10. Next Steps

### Immediate
- [ ] **Wire to real Lens output** — load `results_bundle.json` and run on actual IOI experiments
- [ ] **HITL checkpoint** — interrupt before `generate_followups` so user can edit/approve gaps
- [ ] **Feed follow-ups back into Lens** — convert `FollowUpSpec` → `ExperimentSpec` and extend the bundle

### Loop-back integration
The pipeline can become iterative:
```
Lens → Quill → (HITL: approve follow-ups?) → Lens (round 2) → Quill → ...
```
Cap at N rounds in the orchestrator to avoid infinite loops.

### Phase 2
- [ ] **Literature grounding** — check follow-up suggestions against Scout's research plan (avoid re-discovering known results)
- [ ] **Confidence calibration** — track across Lens runs whether Quill's flagged weaknesses were confirmed
- [ ] **Domain profiles** — different critique profiles for different research types (circuit ID vs. feature analysis vs. probing)